# Multi-modal Deepfake Detection - Exploration Notebook
Trying to see if combining spatial features (standard ResNet) with frequency-domain features (DCT) helps catch AI-generated faces better than just looking at the pixels directly.

Main questions I want to check first:
1. Do the datasets actually load and what do they look like?
2. Does the average DCT spectrum actually show a difference between real FFHQ faces and StyleGAN faces?
3. Quick prototype of the model and sanity check on a small batch.

In [1]:
import os
import glob
import random
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.models as models
import torch_dct

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"using device: {device}")

using device: cuda


In [2]:
# quick sanity check, does the data directory exist?
data_dir = "data/raw/stylegan/real_vs_fake/real-vs-fake"
print("checking splits:", os.listdir(data_dir))

for split in ["train", "valid", "test"]:
    n_real = len(glob.glob(f"{data_dir}/{split}/real/*.jpg"))
    n_fake = len(glob.glob(f"{data_dir}/{split}/fake/*.jpg"))
    print(f"{split} -> real: {n_real}, fake: {n_fake}")

checking splits: ['test', 'train', 'valid']
train -> real: 50000, fake: 50000
valid -> real: 10000, fake: 10000
test -> real: 10000, fake: 10000


In [3]:
# just load one image to see what size it is
sample_path = glob.glob(f"{data_dir}/train/real/*.jpg")[0]
sample_img = Image.open(sample_path)
print("sample image size:", sample_img.size, "mode:", sample_img.mode)
# ok cool, 256x256 RGB already, don't need heavy resizing for stylegan

sample image size: (256, 256) mode: RGB


In [4]:
# let's look at a few real and fake samples side by side
fig, axes = plt.subplots(2, 4, figsize=(12, 6))

real_samples = glob.glob(f"{data_dir}/train/real/*.jpg")[:4]
fake_samples = glob.glob(f"{data_dir}/train/fake/*.jpg")[:4]

for i, p in enumerate(real_samples):
    axes[0, i].imshow(Image.open(p))
    axes[0, i].set_title("Real (FFHQ)")
    axes[0, i].axis("off")

for i, p in enumerate(fake_samples):
    axes[1, i].imshow(Image.open(p))
    axes[1, i].set_title("Fake (StyleGAN)")
    axes[1, i].axis("off")

plt.tight_layout()
plt.show()
# honestly by eye they both look pretty convincing... can't easily spot artifacts at 256x256

### Why DCT instead of FFT?

When looking at image frequency spectrums, the standard approach in a lot of papers is 2D FFT. But 2D FFT implicitly assumes periodic tiling (wrapping around edges). Because the top edge doesn't match the bottom edge, and left doesn't match right, this artificial boundary jump creates a bright "cross" artifact right through the center of every spectrum, real or fake, just from image edges not wrapping smoothly. That cross has nothing to do with whether the image is synthetic or real.

DCT (Discrete Cosine Transform) uses even symmetric reflection at the boundaries rather than periodic wrapping, which eliminates the boundary discontinuity cross. Also, DCT naturally packs energy with the DC/low-frequency component in the top-left corner by convention without needing an `fftshift` step.

Let's compute the mean log-magnitude DCT spectrum across real and fake images to see what shows up in the difference panel.

In [5]:
# compute mean 2D DCT for real vs fake images
def compute_dct_spectrum(img_path):
    img = Image.open(img_path).convert("L") # grayscale
    img_t = T.ToTensor()(img) # [1, H, W] in [0, 1]
    dct = torch_dct.dct_2d(img_t)
    # log1p of magnitude
    mag = torch.log1p(torch.abs(dct)).squeeze(0).numpy()
    return mag

n_spec = 200
real_files = glob.glob(f"{data_dir}/train/real/*.jpg")[:n_spec]
fake_files = glob.glob(f"{data_dir}/train/fake/*.jpg")[:n_spec]

real_spectra = [compute_dct_spectrum(p) for p in real_files]
fake_spectra = [compute_dct_spectrum(p) for p in fake_files]

mean_real_dct = np.mean(real_spectra, axis=0)
mean_fake_dct = np.mean(fake_spectra, axis=0)
diff_dct = mean_fake_dct - mean_real_dct

In [6]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

im0 = axes[0].imshow(mean_real_dct, cmap="magma")
axes[0].set_title("Mean Log-Magnitude DCT (Real FFHQ)")
axes[0].set_xlabel("Horizontal Frequency")
axes[0].set_ylabel("Vertical Frequency")
plt.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)

im1 = axes[1].imshow(mean_fake_dct, cmap="magma")
axes[1].set_title("Mean Log-Magnitude DCT (Fake StyleGAN)")
axes[1].set_xlabel("Horizontal Frequency")
axes[1].set_ylabel("Vertical Frequency")
plt.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)

vmax = max(abs(diff_dct.min()), abs(diff_dct.max()))
im2 = axes[2].imshow(diff_dct, cmap="coolwarm", vmin=-vmax, vmax=vmax)
axes[2].set_title("Difference (Fake - Real)")
axes[2].set_xlabel("Horizontal Frequency")
axes[2].set_ylabel("Vertical Frequency")
plt.colorbar(im2, ax=axes[2], fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()

# notice: no bright cross through the middle! DCT avoids that edge wrap artifact.
# in the difference panel, StyleGAN has subtle high-frequency ripple patterns.

### Building the Detector Architecture
Now we build the `Detector` class. We need:
1. Spatial branch: fine-tuned ResNet-18 (fc replaced with Identity, output 512-d).
2. Frequency branch: grayscale -> 2D DCT -> log1p -> per-image norm -> 5-block CNN from scratch -> 128-d.
3. LayerNorm on both branches before concatenation so spatial features don't drown out frequency features.
4. Separate auxiliary heads for each branch so they get direct supervision even during fusion.
5. Random branch dropout (~25% each) during fusion training so the model doesn't just rely 100% on spatial.

In [7]:
IMG_SIZE = 256

class Detector(nn.Module):
    def __init__(self, mode="fusion"):
        super().__init__()
        self.mode = mode
        
        # spatial branch: pretrained resnet18, fine-tuned (transfer learning piece)
        if mode in ["spatial", "fusion"]:
            weights = models.ResNet18_Weights.DEFAULT
            self.spatial_net = models.resnet18(weights=weights)
            self.spatial_net.fc = nn.Identity()
            # without this the spatial branch's bigger numbers were probably just drowning out freq
            self.spatial_norm = nn.LayerNorm(512)
            # adding this so the spatial branch gets direct gradient signal even in fusion mode
            self.spatial_head = nn.Linear(512, 1)
            
            # imagenet stats for normalization
            self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
            self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

        # freq branch: grayscale -> 2D DCT -> log1p -> normalize -> 5-conv-block CNN from scratch
        # using dct instead of fft, avoids that weird cross artifact fft spectrums always have
        # no pretrained model does dct input so this one's starting from zero
        if mode in ["freq", "fusion"]:
            self.freq_net = nn.Sequential(
                nn.Conv2d(1, 16, 3, padding=1), nn.BatchNorm2d(16), nn.ReLU(), nn.MaxPool2d(2, 2),
                nn.Conv2d(16, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2, 2),
                nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2, 2),
                nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2, 2),
                nn.Conv2d(128, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
                nn.AdaptiveAvgPool2d((1, 1)),
                nn.Flatten()
            )
            # LayerNorm on freq branch too
            self.freq_norm = nn.LayerNorm(128)
            # adding this so the freq branch doesn't just get ignored, trying it out
            self.freq_head = nn.Linear(128, 1)
            
        if mode == "fusion":
            self.fusion_head = nn.Sequential(
                nn.Linear(512 + 128, 128),
                nn.ReLU(),
                nn.Dropout(0.3),
                nn.Linear(128, 1)
            )

    def extract_spatial(self, x):
        # normalize with imagenet mean/std
        x_norm = (x - self.mean) / self.std
        feat = self.spatial_net(x_norm)
        return self.spatial_norm(feat)

    def extract_freq(self, x):
        # grayscale conversion
        gray = 0.2989 * x[:, 0:1] + 0.5870 * x[:, 1:2] + 0.1140 * x[:, 2:3]
        # 2D DCT (low freq is top-left, no fftshift needed)
        dct = torch_dct.dct_2d(gray)
        mag = torch.log1p(torch.abs(dct))
        # normalize per image
        m = mag.mean(dim=(-2, -1), keepdim=True)
        s = mag.std(dim=(-2, -1), keepdim=True) + 1e-6
        mag_norm = (mag - m) / s
        feat = self.freq_net(mag_norm)
        return self.freq_norm(feat)

    def forward(self, x):
        spatial_out, freq_out = None, None
        
        if self.mode == "spatial":
            s_feat = self.extract_spatial(x)
            main_out = self.spatial_head(s_feat)
            return main_out, main_out, None
            
        elif self.mode == "freq":
            f_feat = self.extract_freq(x)
            main_out = self.freq_head(f_feat)
            return main_out, None, main_out
            
        elif self.mode == "fusion":
            s_feat = self.extract_spatial(x)
            f_feat = self.extract_freq(x)
            
            # side outputs for direct gradient signal even in fusion mode
            spatial_out = self.spatial_head(s_feat)
            freq_out = self.freq_head(f_feat)
            
            # forcing it to sometimes work without spatial, see if freq branch actually learns something
            s_in, f_in = s_feat, f_feat
            if self.training:
                r = torch.rand(1).item()
                if r < 0.25:
                    s_in = torch.zeros_like(s_feat)
                elif r < 0.50:
                    f_in = torch.zeros_like(f_feat)
                    
            fused = torch.cat([s_in, f_in], dim=1)
            main_out = self.fusion_head(fused)
            return main_out, spatial_out, freq_out

In [8]:
# quick sanity check, ignore this
dummy_x = torch.randn(4, 3, 256, 256)
test_model = Detector(mode="fusion")
m, s, f = test_model(dummy_x)
print("fusion forward pass check:", m.shape, s.shape, f.shape)

fusion forward pass check: torch.Size([4, 1]) torch.Size([4, 1]) torch.Size([4, 1])


In [9]:
# quick 1-epoch sanity check run on a tiny subset just to see loss go down
# real training runs will happen in train.py!

class MiniDataset(Dataset):
    def __init__(self, paths, labels):
        self.paths = paths
        self.labels = labels
        self.tf = T.Compose([
            T.Resize((256, 256)),
            T.ToTensor()
        ])
        
    def __len__(self):
        return len(self.paths)
        
    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert("RGB")
        return self.tf(img), torch.tensor([self.labels[idx]], dtype=torch.float32)

sample_real = glob.glob(f"{data_dir}/train/real/*.jpg")[:100]
sample_fake = glob.glob(f"{data_dir}/train/fake/*.jpg")[:100]
paths = sample_real + sample_fake
labels = [0.0]*100 + [1.0]*100

mini_ds = MiniDataset(paths, labels)
mini_loader = DataLoader(mini_ds, batch_size=16, shuffle=True)

mini_model = Detector(mode="fusion").to(device)
opt = torch.optim.AdamW(mini_model.parameters(), lr=1e-4)
bce = nn.BCEWithLogitsLoss()

mini_model.train()
print("running quick mini-epoch...")
losses = []
for batch_idx, (bx, by) in enumerate(mini_loader):
    bx, by = bx.to(device), by.to(device)
    opt.zero_grad()
    m_out, s_out, f_out = mini_model(bx)
    loss = bce(m_out, by) + 0.3 * bce(s_out, by) + 0.3 * bce(f_out, by)
    loss.backward()
    opt.step()
    losses.append(loss.item())

print(f"loss started at {losses[0]:.4f} and dropped to {losses[-1]:.4f}")
print("sanity check passed! ready to move Detector to model.py and build train.py")

running quick mini-epoch...
loss started at 1.1245 and dropped to 0.4132
sanity check passed! ready to move Detector to model.py and build train.py
